# Объединение CSV Equaring (2023–2025)

Сливает 6 полугодовых файлов в один `merged_2023_2025.csv`.

Базовые проверки:
- одинаковый набор колонок во всех файлах,
- итоговое число строк после `concat` равно сумме строк по файлам.

In [ ]:
import os
import pandas as pd

DATA_DIR = "/home/jovyan/documents/Equaring/Data"
FILES = [
    "jan-jun2023.csv",
    "jul-dec2023.csv",
    "jan-jun2024.csv",
    "jul-dec2024.csv",
    "jan-jun2025.csv",
    "jul-dec2025.csv",
]
OUT_FILE = f"{DATA_DIR}/merged_2023_2025.csv"

print("DATA_DIR:", DATA_DIR)
print("OUT_FILE:", OUT_FILE)
print("Файлов:", len(FILES))

In [ ]:
# В первых 75 строках каждого файла лежит скрипт выгрузки,
# реальные данные начинаются с 76-й строки (она содержит заголовки колонок).
SKIP_ROWS = 75

dfs = []
stats = []

for name in FILES:
    path = f"{DATA_DIR}/{name}"
    if not os.path.exists(path):
        raise FileNotFoundError(f"Файл не найден: {path}")

    df = pd.read_csv(path, dtype=str, low_memory=False, skiprows=SKIP_ROWS)
    dfs.append(df)
    stats.append({"файл": name, "строк": len(df), "колонок": df.shape[1]})
    print(f"{name}: {len(df):,} строк, {df.shape[1]} колонок")

stats_df = pd.DataFrame(stats)
print("\nИтого по файлам:")
display(stats_df)

In [ ]:
base_cols = set(dfs[0].columns)
mismatch = {}
for name, df in zip(FILES, dfs):
    cols = set(df.columns)
    only_here = cols - base_cols
    only_in_base = base_cols - cols
    if only_here or only_in_base:
        mismatch[name] = {
            "лишние": sorted(only_here),
            "отсутствуют": sorted(only_in_base),
        }

if mismatch:
    print("Расхождения по колонкам относительно первого файла:")
    for name, diff in mismatch.items():
        print(f"\n{name}:")
        print(f"  лишние:      {diff['лишние']}")
        print(f"  отсутствуют: {diff['отсутствуют']}")
    raise AssertionError("Набор колонок различается между файлами — слияние остановлено.")

print("Все файлы имеют одинаковый набор колонок:", len(base_cols), "шт.")

expected_rows = sum(len(df) for df in dfs)
print(f"Ожидаемое число строк после concat: {expected_rows:,}")

In [ ]:
merged = pd.concat(dfs, ignore_index=True)

assert len(merged) == expected_rows, (
    f"После concat строк: {len(merged):,}, ожидалось: {expected_rows:,}"
)

merged.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("Готово.")
print(f"Файл:    {OUT_FILE}")
print(f"Строк:   {len(merged):,}")
print(f"Колонок: {merged.shape[1]}")